# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. **`20 10`** — у Series `s` ярлык `"b"` соответствует числу 20, а `iloc[0]` берёт первое значение, то есть 10.
2. **`Алматы: NaN`, `Астана: NaN`, `Шымкент: 3.0`** — Pandas складывает Series по названиям индексов, а не по порядку. Только Шымкент есть в обоих: 2 + 1 = 3. Для остальных не хватает одного слагаемого.
3. **`(2, 2)`** — из трёх строк остаются две (где `x > 1`), и обе колонки сохраняются.
4. **`Series DataFrame`** — один столбец возвращает Series, а двойные скобки со списком имён столбцов возвращают таблицу.
5. **`[1, 2] 5`** — фильтр оставил исходные индексы строк 1 и 2. `iloc[0, 1]` обращается к первой оставшейся строке и второму столбцу `y`, где стоит 5.


In [3]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [5]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [6]:
# 1. Паспорт таблицы
print("Размер таблицы:", df.shape)
print("Типы столбцов:")
print(df.dtypes)
print("Тип amount в исходном CSV:", pd.read_csv("orders.csv")["amount"].dtype)
print("Пропусков delivery_days:", len(df) - df["delivery_days"].count())
print("Написаний city:", df["city"].nunique())
print("Категорий:", df["category"].nunique())
print("Статусов:", df["status"].nunique())
print("Клиентов:", df["customer_id"].nunique())
print("Повторных order_id:", len(df) - df["order_id"].nunique())
print("\nМенеджеру: в CSV суммы были текстом, поэтому без преобразования выручка не считается как число.")
print("Города имеют 14 написаний: без объединения названий статистика по городам неверна.")
print("Есть 180 повторов order_id: они могут завышать выручку; в доставке 160 пропусков.")


Размер таблицы: (5463, 8)
Типы столбцов:
order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
Тип amount в исходном CSV: object
Пропусков delivery_days: 160
Написаний city: 14
Категорий: 5
Статусов: 2
Клиентов: 2003
Повторных order_id: 180

Менеджеру: в CSV суммы были текстом, поэтому без преобразования выручка не считается как число.
Города имеют 14 написаний: без объединения названий статистика по городам неверна.
Есть 180 повторов order_id: они могут завышать выручку; в доставке 160 пропусков.


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [7]:
# 2. Типичный заказ: только оплаченные и без клиента TEST
checks = df.loc[paid & not_test, "amount"]
print("Оплаченных заказов без TEST:", len(checks))
print("Среднее:", round(checks.mean(), 2), "₸")
print("Медиана:", checks.median(), "₸")
print("25-й перцентиль:", checks.quantile(0.25), "₸")
print("75-й перцентиль:", checks.quantile(0.75), "₸")
print("Менеджеру: типичный чек около 10 300 ₸. Лучше показать медиану:")
print("дорогие заказы сдвигают среднее вверх, а медиана менее чувствительна к ним.")


Оплаченных заказов без TEST: 4901
Среднее: 14672.39 ₸
Медиана: 10300.0 ₸
25-й перцентиль: 5880.0 ₸
75-й перцентиль: 18690.0 ₸
Менеджеру: типичный чек около 10 300 ₸. Лучше показать медиану:
дорогие заказы сдвигают среднее вверх, а медиана менее чувствительна к ним.


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [8]:
# 3. Отмены по городам (учитываем все известные написания из CSV)
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
sh_mask = df["city"].isin(SH)
al_mask = df["city"].isin(AL)
cancel = df["status"] == "отменён"

all_rate = cancel.mean() * 100
sh_rate = cancel[sh_mask].mean() * 100
al_rate = cancel[al_mask].mean() * 100
print("Отмены во всех городах:", round(all_rate, 2), "%")
print("Шымкент:", sh_mask.sum(), "заказа; отмены", round(sh_rate, 2), "%")
print("Алматы:", al_mask.sum(), "заказов; отмены", round(al_rate, 2), "%")
print("Отмены в Шымкенте чаще, чем в Алматы, в", round(sh_rate / al_rate, 2), "раза")
print("Менеджеру: проверить причины высокого процента отмен в Шымкенте.")


Отмены во всех городах: 10.23 %
Шымкент: 804 заказа; отмены 20.4 %
Алматы: 2188 заказов; отмены 7.86 %
Отмены в Шымкенте чаще, чем в Алматы, в 2.59 раза
Менеджеру: проверить причины высокого процента отмен в Шымкенте.


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [9]:
# 4. Выручка двух городов по месяцам: оплаченные без TEST, дубли пока оставляем
sh_sep = df.loc[paid & not_test & sep & sh_mask, "amount"].sum()
sh_oct = df.loc[paid & not_test & octo & sh_mask, "amount"].sum()
al_sep = df.loc[paid & not_test & sep & al_mask, "amount"].sum()
al_oct = df.loc[paid & not_test & octo & al_mask, "amount"].sum()
sh_change = (sh_oct / sh_sep - 1) * 100
al_change = (al_oct / al_sep - 1) * 100
print("Шымкент: сентябрь", sh_sep, "₸; октябрь", sh_oct, "₸; изменение", round(sh_change, 2), "%")
print("Алматы: сентябрь", al_sep, "₸; октябрь", al_oct, "₸; изменение", round(al_change, 2), "%")
print("Падение в Шымкенте сильнее в", round(sh_change / al_change, 2), "раза")
print("Менеджеру: обратить внимание на Шымкент, но для точного отчёта ещё убрать повторы order_id.")


Шымкент: сентябрь 4949800.0 ₸; октябрь 4044010.0 ₸; изменение -18.3 %
Алматы: сентябрь 15523940.0 ₸; октябрь 14643020.0 ₸; изменение -5.67 %
Падение в Шымкенте сильнее в 3.22 раза
Менеджеру: обратить внимание на Шымкент, но для точного отчёта ещё убрать повторы order_id.


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [10]:
# 5. Исправление двух ошибок в коде AI
# Ошибка 1: нет скобок вокруг двух условий при использовании &.
# Ошибка 2: сравнение только с "Шымкент" теряет написания "шымкент" и "Shymkent".
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print("Отменённых заказов Шымкента:", len(cancel_sh))
print("Менеджеру: учитываем все три написания города, иначе число отмен будет занижено.")


Отменённых заказов Шымкента: 164
Менеджеру: учитываем все три написания города, иначе число отмен будет занижено.


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [11]:
# 6. Самый частый клиент и повторы заказов
client_counts = df["customer_id"].value_counts()
print("Чаще всего встречается:")
print(client_counts.head(1))
most_frequent = client_counts.index[0]  # индекс первой строки: клиент с максимумом заказов
client_orders = df.loc[df["customer_id"] == most_frequent,
                       ["order_date", "city", "amount", "status", "order_id"]]
print("\nЗаказы клиента", most_frequent, ":")
print(client_orders)
repeat_count = len(client_orders) - client_orders["order_id"].nunique()
print("Повторных строк с одним order_id у этого клиента:", repeat_count)
print("Менеджеру: один заказ может встречаться несколько раз и завышать выручку.")


Чаще всего встречается:
customer_id
C01818    11
Name: count, dtype: int64

Заказы клиента C01818 :
            order_date       city   amount   status  order_id
57    2026-09-01 15:00    Шымкент   7350.0  оплачен    101125
779   2026-09-09 18:00    Шымкент  45270.0  оплачен    100096
1371  2026-09-15 21:00     Алматы   8890.0  оплачен    101504
1375  2026-09-15 21:00     Алматы   8890.0  оплачен    101504
2195  2026-09-24 18:00     Астана   1490.0  оплачен    101860
2617  2026-09-29 09:00     Астана  22170.0  оплачен    101462
2776  2026-09-30 21:00     Алматы  20180.0  оплачен    101495
3417  2026-10-08 14:00  Караганда   4330.0  оплачен    103416
4279  2026-10-18 08:00     Астана  13600.0  оплачен    102644
4610  2026-10-21 19:00     Алматы  21990.0  оплачен    104001
4778  2026-10-23 18:00     Алматы  11140.0  оплачен    104572
Повторных строк с одним order_id у этого клиента: 1
Менеджеру: один заказ может встречаться несколько раз и завышать выручку.


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [12]:
# 7. IQR: ищем слишком дорогие оплаченные заказы без TEST
q1 = checks.quantile(0.25)
q3 = checks.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers = df[paid & not_test & (df["amount"] > upper)]
print("Q1:", q1, "Q3:", q3, "IQR:", iqr, "верхняя граница:", upper)
print("Выбросов по IQR:", len(outliers))
print("Доля от оплаченных без TEST:", round(len(outliers) / len(checks) * 100, 2), "%")
print("Категории выбросов:")
print(outliers["category"].value_counts())
print("Менеджеру: чаще всего это электроника. По z-оценке в ДЗ 4 было 87 заказов.")
print("IQR смотрит на квартильный разброс, а z-оценка — на среднее и отклонение,")
print("поэтому при асимметричном распределении количества различаются.")


Q1: 5880.0 Q3: 18690.0 IQR: 12810.0 верхняя граница: 37905.0
Выбросов по IQR: 320
Доля от оплаченных без TEST: 6.53 %
Категории выбросов:
category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64
Менеджеру: чаще всего это электроника. По z-оценке в ДЗ 4 было 87 заказов.
IQR смотрит на квартильный разброс, а z-оценка — на среднее и отклонение,
поэтому при асимметричном распределении количества различаются.


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [13]:
# 8. Две Series автоматически сопоставляются по названиям городов
cancel_by_city = df.loc[df["status"] == "отменён", "city"].value_counts()
all_by_city = df["city"].value_counts()
cancel_part = cancel_by_city / all_by_city
print("Доля отмен по написаниям городов (от большего к меньшему):")
print((cancel_part * 100).sort_values(ascending=False))
print("\nЗаказы трёх написаний Шымкента:")
print(all_by_city[["Шымкент", "шымкент", "Shymkent"]])
print("Доли отмен в этих написаниях:")
print((cancel_part[["Шымкент", "шымкент", "Shymkent"]] * 100))
print("Менеджеру: по одной записи города судить нельзя — надо привести названия")
print("к одному виду перед расчётом (очистка городов будет на уроке 6).")


Доля отмен по написаниям городов (от большего к меньшему):
city
Shymkent     23.076923
Шымкент      21.518987
алматы       14.062500
астана       13.095238
Актобе       10.327456
Astana       10.000000
Караганда     8.921933
Астана        8.482871
Алматы        7.991697
Атырау        7.534247
Almaty        5.797101
 Алматы       4.545455
шымкент       3.703704
АЛМАТЫ        3.225806
Name: count, dtype: float64

Заказы трёх написаний Шымкента:
city
Шымкент     711
шымкент      54
Shymkent     39
Name: count, dtype: int64
Доли отмен в этих написаниях:
city
Шымкент     21.518987
шымкент      3.703704
Shymkent    23.076923
Name: count, dtype: float64
Менеджеру: по одной записи города судить нельзя — надо привести названия
к одному виду перед расчётом (очистка городов будет на уроке 6).
